In [ ]:

# 检查并安装LightGBM和pandas包
try:
    import lightgbm as lgb
    import pandas as pd
    import numpy as np
    from sklearn.metrics import roc_auc_score
except ImportError:
    !pip install lightgbm pandas numpy scikit-learn


In [ ]:


# Load the training and testing datasets
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/train.csv'
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/reservation_cancellation/test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# Display the first few rows of the training and testing datasets
print(train_df.head())
print(test_df.head())



      id  no_of_adults  ...  no_of_special_requests  booking_status
0  15559             2  ...                       2               0
1  32783             2  ...                       1               0
2  11797             3  ...                       0               1
3  39750             2  ...                       1               1
4  28711             2  ...                       0               1

[5 rows x 19 columns]
      id  no_of_adults  ...  no_of_special_requests  booking_status
0   8768             2  ...                       1               0
1  38340             2  ...                       0               1
2   7104             2  ...                       0               0
3  36898             2  ...                       3               0
4   9747             2  ...                       1               0

[5 rows x 19 columns]


In [ ]:



# Check for missing values in the training and testing datasets
print(train_df.isnull().sum())
print(test_df.isnull().sum())




id                                      0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights                       0
type_of_meal_plan                       0
required_car_parking_space              0
room_type_reserved                      0
lead_time                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
market_segment_type                     0
repeated_guest                          0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
avg_price_per_room                      0
no_of_special_requests                  0
booking_status                          0
dtype: int64
id                                      0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights    

In [ ]:



# Encode categorical variables in the training dataset
categorical_cols = train_df.select_dtypes(include=['object']).columns
train_df_encoded = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)

# Encode categorical variables in the testing dataset
test_df_encoded = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Ensure both datasets have the same columns
missing_cols = set(train_df_encoded.columns) - set(test_df_encoded.columns)
for col in missing_cols:
    test_df_encoded[col] = 0

test_df_encoded = test_df_encoded[train_df_encoded.columns]

# Split the training data into features (X_train) and target variable (y_train)
X_train = train_df_encoded.drop(columns=['booking_status', 'id'])
y_train = train_df_encoded['booking_status']

# Split the testing data into features (X_test)
X_test = test_df_encoded.drop(columns=['booking_status', 'id'])

# Display the first few rows of the encoded training data
print(X_train.head())
print(y_train.head())

# Display the first few rows of the encoded testing data
print(X_test.head())






   no_of_adults  no_of_children  ...  avg_price_per_room  no_of_special_requests
0             2               0  ...                60.0                       2
1             2               0  ...               111.6                       1
2             3               0  ...               168.3                       0
3             2               0  ...               115.0                       1
4             2               0  ...               132.3                       0

[5 rows x 17 columns]
0    0
1    0
2    1
3    1
4    1
Name: booking_status, dtype: int64
   no_of_adults  no_of_children  ...  avg_price_per_room  no_of_special_requests
0             2               0  ...               80.67                       1
1             2               0  ...               93.96                       0
2             2               0  ...              104.65                       0
3             2               1  ...              124.67                       3
4             2 

In [ ]:



# Train the LightGBM model
lgb_train = lgb.Dataset(X_train, label=y_train)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',  
    'max_depth': 8,     
    'learning_rate': 0.1,     
    'n_leaves': 50,
}

num_rounds = 50

gbm = lgb.train(params, lgb_train, num_rounds)

# Make predictions on the test dataset
y_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)

# Display the first few predictions
print(y_pred[:10])  # Print the first 10 predictions for verification



[LightGBM] [Warning] Unknown parameter: n_leaves
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=8) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=256) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Unknown parameter: n_leaves
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=8) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=256) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Info] Number of positive: 13233, number of negative: 20441
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001572 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[

In [ ]:



# Adjust parameters to use num_leaves instead of n_leaves
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',  
    'max_depth': 8,     
    'learning_rate': 0.1,     
    'num_leaves': 50,
}

# Train the LightGBM model again
lgb_train = lgb.Dataset(X_train, label=y_train)
num_rounds = 50

gbm = lgb.train(params, lgb_train, num_rounds)

# Make predictions on the test dataset
y_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)

# Display the first few predictions
print(y_pred[:10])  # Print the first 10 predictions for verification




[LightGBM] [Info] Number of positive: 13233, number of negative: 20441
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009404 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 652
[LightGBM] [Info] Number of data points in the train set: 33674, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.392974 -> initscore=-0.434829
[LightGBM] [Info] Start training from score -0.434829
[0.01794519 0.87783276 0.39943425 0.01390198 0.18276848 0.00967549
 0.74596246 0.42723371 0.7772903  0.78398547]


In [ ]:




# Save predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'booking_status': y_pred})
predictions_df.to_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions/reservation_cancellation_predictions.csv', index=False)

# Display the first few rows of the saved predictions
print(predictions_df.head())






Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

rminator,
   3970     sep=sep,
   3971     encoding=encoding,
   3972     errors=errors,
   3973     compression=compression,
   3974     quoting=quoting,
   3975     columns=columns,
   3976     index_label=index_label,
   3977     mode=mode,
   3978     chunksize=chunksize,
   3979     quotechar=quotechar,
   3980     date_format=date_format,
   3981     doublequote=doublequote,
   3982     escapechar=escapechar,
   3983     storage_options=storage_options,
   3984 )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\io\formats\format.py:1014, in DataFrameRenderer.to_csv(self, path_or_buf, encoding, sep, columns, index_label, mode, compression, quoting, quotechar, lineterminator, chunksize, date_format, doublequote, escapechar, errors, storage_options)
    993     created_buff

In [ ]:




# Create the directory if it does not exist
import os
predictions_dir = r'D:\LLM-Driven_AI-Studio\MLAgent\data\predictions'
os.makedirs(predictions_dir, exist_ok=True)

# Save predictions to a CSV file
predictions_df = pd.DataFrame({'id': test_df['id'], 'booking_status': y_pred})
predictions_df.to_csv(os.path.join(predictions_dir, 'reservation_cancellation_predictions.csv'), index=False)

# Display the first few rows of the saved predictions
print(predictions_df.head())





      id  booking_status
0   8768        0.017945
1  38340        0.877833
2   7104        0.399434
3  36898        0.013902
4   9747        0.182768


In [ ]:





# Calculate the AUC-ROC score
# Assuming y_train contains the true labels and y_pred contains the predicted probabilities
y_true = train_df['booking_status']
y_train_pred = gbm.predict(X_train, num_iteration=gbm.best_iteration)

train_auc_roc = roc_auc_score(y_true, y_train_pred)

# Calculate the AUC-ROC score on the test set
y_test_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)
test_auc_roc = roc_auc_score(y_test, y_test_pred)

# Print the AUC-ROC scores
print(f"Train AUC-ROC: {train_auc_roc:.4f}")
print(f"Test AUC-ROC: {test_auc_roc:.4f}")




---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[10], line 20
     18 y_test_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)
     19 print('##active_line10##')
---> 20 test_auc_roc = roc_auc_score(y_test, y_test_pred)
     21 print('##active_line11##')
     22 pass

NameError: name 'y_test' is not defined

In [ ]:






# Extract the true labels from the test dataset
y_test = test_df['booking_status']

# Calculate the AUC-ROC score
# Assuming y_train contains the true labels and y_pred contains the predicted probabilities
y_true = train_df['booking_status']
y_train_pred = gbm.predict(X_train, num_iteration=gbm.best_iteration)

train_auc_roc = roc_auc_score(y_true, y_train_pred)

# Calculate the AUC-ROC score on the test set
y_test_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)
test_auc_roc = roc_auc_score(y_test, y_test_pred)

# Print the AUC-ROC scores
print(f"Train AUC-ROC: {train_auc_roc:.4f}")
print(f"Test AUC-ROC: {test_auc_roc:.4f}")





Train AUC-ROC: 0.9048
Test AUC-ROC: 0.8926


In [ ]:






# Save the trained LightGBM model
import joblib
model_path = r'D:\LLM-Driven_AI-Studio\MLAgent\models\lightgbm_model.joblib'
joblib.dump(gbm, model_path)

print(f"Model saved to {model_path}")






---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[12], line 8
      6 model_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\models\\lightgbm_model.joblib'
      7 print('##active_line4##')
----> 8 joblib.dump(gbm, model_path)
      9 print('##active_line5##')
     10 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\joblib\numpy_pickle.py:552, in dump(value, filename, compress, protocol, cache_size)
    550         NumpyPickler(f, protocol=protocol).dump(value)
    551 elif is_filename:
--> 552     with open(filename, 'wb') as f:
    553         NumpyPickler(f, protocol=protocol).dump(value)
    554 else:

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\LLM-Driven_AI-Studio\\MLAgent\\models\\lightgbm_model.joblib'

In [ ]:







# Create the directory if it does not exist
import os
model_dir = r'D:\LLM-Driven_AI-Studio\MLAgent\models'
os.makedirs(model_dir, exist_ok=True)

# Save the trained LightGBM model
import joblib
model_path = r'D:\LLM-Driven_AI-Studio\MLAgent\models\lightgbm_model.joblib'
joblib.dump(gbm, model_path)

print(f"Model saved to {model_path}")








Model saved to D:\LLM-Driven_AI-Studio\MLAgent\models\lightgbm_model.joblib
